# Part 2 — Building a Golden Set

*Evals from First Principles, Part 2.*

Part 1 built the confusion matrix and its four metrics on top of 10 already-labeled tickets. This notebook asks the question those metrics quietly assumed away: where do the gold labels come from, and what makes the set of them trustworthy? We build a golden set by hand on tiny, printable data and hit three traps in order: a rare class that simple random sampling can miss entirely, near-duplicate items that leak from training data and inflate accuracy, and a vague "good answer" bar that a rubric turns into countable criteria.

Pure Python + NumPy, offline and deterministic — every random draw uses a fixed seed (`numpy.random.default_rng(0)`), so every number below is reproducible on a laptop with no network and no API key. The hypergeometric facts are computed exactly with `math.comb` and used to cross-check the simulation, never the other way around.

Companion script: `golden_set.py`

## Step 1 — A rare class and a tight labeling budget

Say we run a support queue of 40 tickets. 8 of them are truly **urgent** and 32 are not — a true urgent rate of 0.20. Hand-labeling is expensive, so we can only afford to label `n=10` tickets for our golden set. How we choose those 10 tickets decides whether the resulting set is trustworthy.

In [ ]:
import math
import numpy as np

SEED = 0
BAR = "=" * 72
SUB = "-" * 72

POP_URGENT = 8
POP_NOT = 32
POP_N = POP_URGENT + POP_NOT          # 40
POP_RATE = POP_URGENT / POP_N         # 0.20 true urgent rate
SAMPLE_N = 10


def make_population():
    """Return the 40 labels: 8 ones (urgent) then 32 zeros (not)."""
    return np.array([1] * POP_URGENT + [0] * POP_NOT)


print(BAR)
print("PART 2 - BUILDING A GOLDEN SET")
print(BAR)
print("Population: 40 tickets, 8 urgent (rare) + 32 not. True urgent rate = 0.20.")
print(f"We can label only n={SAMPLE_N} for the golden set.\n")

## Step 2 — Simple random vs. stratified sampling

**Simple random sampling** draws `n` tickets uniformly from all 40, no matter their label. **Stratified sampling** instead draws from each class separately, keeping each class's share fixed at its population rate — 20% urgent means a sample of 10 always gets exactly 2 urgent and 8 not, drawn without replacement within each stratum. Let's draw one sample of each kind and compare.

In [ ]:
def simple_random_sample(pop, n, rng):
    """Draw n items uniformly WITHOUT replacement from the whole population."""
    idx = rng.choice(len(pop), size=n, replace=False)
    return pop[idx]


def stratified_sample(pop, n, rng):
    """Draw n items keeping each class's share fixed at its population rate.

    urgent share = 0.20, so a sample of 10 gets exactly 2 urgent + 8 not,
    drawn without replacement WITHIN each stratum.
    """
    urgent_idx = np.where(pop == 1)[0]
    not_idx = np.where(pop == 0)[0]
    take_urgent = round(n * POP_URGENT / len(pop))   # 10 * 8/40 = 2
    take_not = n - take_urgent                        # 8
    pick_u = rng.choice(urgent_idx, size=take_urgent, replace=False)
    pick_n = rng.choice(not_idx, size=take_not, replace=False)
    return pop[np.concatenate([pick_u, pick_n])]


rng = np.random.default_rng(SEED)
pop = make_population()

srs = simple_random_sample(pop, SAMPLE_N, rng)
strat = stratified_sample(pop, SAMPLE_N, rng)
print("One SIMPLE-RANDOM sample of 10:")
print(f"  urgent picked = {int(srs.sum())}  ->  measured urgent rate = {srs.mean():.2f}")
print("One STRATIFIED sample of 10 (share fixed at 0.20):")
print(f"  urgent picked = {int(strat.sum())}  ->  measured urgent rate = {strat.mean():.2f}")

## Step 3 — Comparing the spread over many resamples

One draw of each is a single data point, not evidence. Let's repeat both sampling methods 10000 times with a fixed seed and compare the mean and standard deviation of the urgent count each one produces, plus how often the simple-random draw ends up with zero urgent tickets at all.

In [ ]:
trials = 10000
rng = np.random.default_rng(SEED)
srs_counts = np.array([simple_random_sample(pop, SAMPLE_N, rng).sum()
                       for _ in range(trials)])
strat_counts = np.array([stratified_sample(pop, SAMPLE_N, rng).sum()
                         for _ in range(trials)])
srs_zero = int((srs_counts == 0).sum())

print(f"Over {trials} resamples (fixed seed):")
print(f"  simple-random : mean urgent = {srs_counts.mean():.3f}, "
      f"std = {srs_counts.std():.3f}")
print(f"  stratified    : mean urgent = {strat_counts.mean():.3f}, "
      f"std = {strat_counts.std():.3f}")
print(f"  simple-random samples with ZERO urgent: "
      f"{srs_zero}/{trials} = {srs_zero / trials:.3f}")

## Step 4 — Cross-checking against the exact hypergeometric facts

The simulation above is convincing, but we don't have to trust it blindly: drawing 10 tickets without replacement from a population with 8 urgent and 32 not is exactly the hypergeometric distribution, so both `P(zero urgent)` and the standard deviation of the urgent count have closed forms we can compute directly with `math.comb`.

In [ ]:
def prob_zero_urgent_exact():
    """Exact P(a simple random sample of 10 contains ZERO urgent tickets).

    Hypergeometric: choose all 10 from the 32 not-urgent, over all C(40,10).
    """
    return math.comb(POP_NOT, SAMPLE_N) / math.comb(POP_N, SAMPLE_N)


def std_urgent_count_exact():
    """Exact std of the urgent COUNT in a simple random sample of 10.

    Hypergeometric variance: n * p * (1-p) * (N-n)/(N-1).
    """
    p = POP_RATE
    var = SAMPLE_N * p * (1 - p) * (POP_N - SAMPLE_N) / (POP_N - 1)
    return math.sqrt(var)


print("Exact (hypergeometric) cross-check of the simple-random draw:")
print(f"  P(zero urgent)      = C(32,10)/C(40,10) = {prob_zero_urgent_exact():.3f}")
print(f"  std of urgent count = sqrt(n*p*(1-p)*(N-n)/(N-1)) = "
      f"{std_urgent_count_exact():.3f}")
print("  -> stratifying pins the rare-class share, driving its std to 0; simple")
print("     random sampling has real spread and can miss urgent tickets entirely.")

## Step 5 — The leakage trap

A golden set is only honest if the model has never seen it before. Suppose a model was "trained" on 6 support tickets (`TRAIN_TICKETS`). Our eval set of 10 tickets (`EVAL_TICKETS`) should test genuinely unseen cases, but a few of its items are near-duplicates of training tickets — same text up to case, punctuation, or spacing. Normalizing both sides and matching finds those leaks.

In [ ]:
TRAIN_TICKETS = [
    "How do I reset my password?",
    "My invoice is wrong",
    "The app crashes on login",
    "Where is my refund?",
    "Cancel my subscription",
    "Update my billing address",
]

# (eval text, model got it right?) - memorized duplicates are always "right".
EVAL_TICKETS = [
    ("how do i reset my password", 1),    # near-dup of train #1 (memorized)
    ("The app crashes on login.", 1),     # near-dup of train #3 (memorized)
    ("How do I export my data?", 1),      # genuine, correct
    ("Is there a mobile app?", 1),        # genuine, correct
    ("What are your business hours?", 0),  # genuine, wrong
    ("cancel my subscription!", 1),        # near-dup of train #5 (memorized)
    ("How do I change my email?", 1),      # genuine, correct
    ("Do you offer discounts?", 0),        # genuine, wrong
    ("My invoice is wrong.", 1),           # near-dup of train #2 (memorized)
    ("Can I get a receipt?", 1),           # genuine, correct
]


def normalize(text):
    """Lowercase, keep only alphanumerics and spaces, collapse whitespace."""
    kept = "".join(c if c.isalnum() or c.isspace() else " " for c in text.lower())
    return " ".join(kept.split())


def find_leaks(train, eval_set):
    """Return the eval indices whose normalized text also appears in train."""
    train_norm = {normalize(t) for t in train}
    return [i for i, (text, _) in enumerate(eval_set)
            if normalize(text) in train_norm]


print(SUB)
print("LEAKAGE: near-duplicates of training data sneak into the eval set.")
print(SUB)
leaks = find_leaks(TRAIN_TICKETS, EVAL_TICKETS)
print(f"eval size = {len(EVAL_TICKETS)}   train size = {len(TRAIN_TICKETS)}")
print(f"leaked eval items (normalized text found in train): {len(leaks)}")
for i in leaks:
    print(f"  eval[{i}] {EVAL_TICKETS[i][0]!r}  ==  a train ticket")

## Step 6 — Measured vs. honest accuracy

The 4 leaked items are all marked correct — the model memorized them, it didn't reason about them. Let's compare accuracy on the full eval set against accuracy after dropping the leaked items.

In [ ]:
def accuracy(rows):
    return sum(correct for _, correct in rows) / len(rows) if rows else 0.0


with_leak = accuracy(EVAL_TICKETS)
clean = [row for i, row in enumerate(EVAL_TICKETS) if i not in leaks]
honest = accuracy(clean)
leak_correct = sum(EVAL_TICKETS[i][1] for i in leaks)

print(f"measured accuracy WITH leakage  = {sum(c for _, c in EVAL_TICKETS)}"
      f"/{len(EVAL_TICKETS)} = {with_leak:.3f}")
print(f"  (the {len(leaks)} memorized duplicates are all correct: "
      f"{leak_correct}/{len(leaks)})")
print(f"honest accuracy AFTER removing leaks = "
      f"{sum(c for _, c in clean)}/{len(clean)} = {honest:.3f}")
print(f"  -> leakage inflated the score by {with_leak - honest:.3f}. The gimme")
print("     duplicates hid the model's real performance on unseen tickets.")

## Step 7 — From a vague bar to an operational rubric

"Was it a good answer?" is a vibe, not a measurement. A rubric turns it into countable criteria: here, three criteria (Correct, Grounded, Complete), each scored 0 (fails), 1 (partial), or 2 (full). Max score is 6, and an answer passes the golden set at a total of 5 or more.

In [ ]:
RUBRIC_CRITERIA = ["Correct", "Grounded", "Complete"]
PASS_THRESHOLD = 5

# (label, [Correct, Grounded, Complete]) - three graded candidate answers.
CANDIDATES = [
    ("Answer A", [2, 2, 2]),   # right, grounded, complete
    ("Answer B", [2, 2, 1]),   # right and grounded, but partial
    ("Answer C", [1, 0, 2]),   # complete but half-right and ungrounded
]

print(SUB)
print("RUBRIC: turn 'good answer?' into countable criteria (each 0/1/2).")
print(SUB)
print(f"criteria: {', '.join(RUBRIC_CRITERIA)}   (max {2 * len(RUBRIC_CRITERIA)}, "
      f"pass at total >= {PASS_THRESHOLD})")
for label, scores in CANDIDATES:
    total = sum(scores)
    verdict = "PASS" if total >= PASS_THRESHOLD else "FAIL"
    cells = "  ".join(f"{name}={s}" for name, s in zip(RUBRIC_CRITERIA, scores))
    print(f"  {label}: {cells}  ->  total {total}/6  {verdict}")
passing = sum(1 for _, s in CANDIDATES if sum(s) >= PASS_THRESHOLD)
print(f"\n{passing}/{len(CANDIDATES)} answers clear the bar. A vibe check might wave")
print("all three through; the rubric makes the bar a number anyone can re-count.")

## Recap

- **Stratified sampling beats simple random sampling on a rare class.** Over 10000 resamples, simple random sampling's urgent count had mean 1.997 and std 1.105, and missed the urgent class entirely in 7.4% of draws — matching the exact hypergeometric prediction of 0.076. Stratified sampling pinned the urgent count at exactly 2 every single time (std 0.000).
- **Leakage inflates measured accuracy.** 4 of 10 eval tickets were near-duplicates of training tickets the model had memorized. Measured accuracy read 0.800 with those duplicates in the set, but dropped to the honest 0.667 once they were removed — an inflation of 0.133 from gimmes that hid the model's real performance.
- **A rubric turns "good answer?" into a number anyone can re-count.** Three 0/1/2 criteria (Correct, Grounded, Complete) scored Answer A at 6/6, Answer B at 5/6, and Answer C at 3/6 — 2 of 3 pass the bar of >= 5, a verdict a vibe check could not reproduce.

Next: how a metric behaves once you stop looking at a single golden set and start asking how much it would wobble on another one just like it.